In [116]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, ClassLabel

In [117]:
train = pd.read_csv("../dataimport/train.csv")
valid = pd.read_csv("../dataimport/valid.csv")
test = pd.read_csv("../dataimport/test.csv")

print("Train :", train.shape)
print("Valid :", valid.shape)
print("Test  :", test.shape)

Train : (10240, 14)
Valid : (1284, 14)
Test  : (1267, 14)


statement_len_words : Nombre de mots dans le statement
statement_len_chars : Nombre de caractères dans le statement

In [118]:
train["statement_len_words"] = train["statement"].astype(str).apply(lambda x: len(x.split()))
valid["statement_len_words"] = valid["statement"].astype(str).apply(lambda x: len(x.split()))
test["statement_len_words"] = test["statement"].astype(str).apply(lambda x: len(x.split()))

train["statement_len_chars"] = train["statement"].astype(str).apply(len)
valid["statement_len_chars"] = valid["statement"].astype(str).apply(len)
test["statement_len_chars"] = test["statement"].astype(str).apply(len)

nb_exclamation : Nombre de points d'exclamation !
nb_question : Nombre de points d'interrogation ?

In [119]:
train["nb_exclamation"] = train["statement"].astype(str).apply(lambda x: x.count("!"))
valid["nb_exclamation"] = valid["statement"].astype(str).apply(lambda x: x.count("!"))
test["nb_exclamation"] = test["statement"].astype(str).apply(lambda x: x.count("!"))

train["nb_question"] = train["statement"].astype(str).apply(lambda x: x.count("?"))
valid["nb_question"] = valid["statement"].astype(str).apply(lambda x: x.count("?"))
test["nb_question"] = test["statement"].astype(str).apply(lambda x: x.count("?"))

Creation len word / nb_exclamation	/ nb_question

In [120]:
train.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,statement_len_words,statement_len_chars,nb_exclamation,nb_question
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,11,82,0,0
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,24,141,0,1
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,19,105,0,0
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,12,78,0,0
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,10,54,0,0


In [121]:
count_cols = [
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts"
]

for df in [train, valid, test]:
    for col in count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["history_total"] = df[count_cols].sum(axis=1)
    df["history_false_ratio"] = (df["false_counts"] + df["pants_on_fire_counts"]) / (df["history_total"] + 1)

^ passif du locuteur history_total	

In [122]:
df.head() 

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,statement_len_words,statement_len_chars,nb_exclamation,nb_question,history_total,history_false_ratio
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,42,23,18,Radio interview,11,68,0,0,143,0.333333
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,0,0,0,a news conference,12,63,0,0,3,0.250000
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,51,37,61,comments on ABC's This Week.,10,51,0,0,326,0.535168
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,3,1,1,a radio show,13,85,0,0,7,0.250000
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,2,2,7,a web video,23,127,0,0,23,0.583333


In [123]:
import torch
print("CUDA available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

CUDA available: False count: 0
Device: cpu


In [124]:
def normalize_text(t):
    return str(t).strip().lower()  # customiser
for df in [train, valid, test]:
    df["statement_clean"] = df["statement"].astype(str).apply(normalize_text)

In [125]:
# affichage des labels actuels pour debug
print("train label uniques:", train["label"].value_counts().to_dict())
print("valid label uniques:", valid["label"].value_counts().to_dict())
print("test label uniques:", test["label"].value_counts().to_dict())

# fonction de mapping plus robuste (toutes les variantes possibles)
def to_binary_label(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower().replace(" ", "_")
    if s in {"fake", "false", "pants_on_fire", "pants_on_fire_counts", "pants-on-fire", "barely_true", "barely_true", "half_true", "half-true", "mostly_true"}:
        return 0
    if s in {"real", "true", "mostly_true", "true_", "true", "mostly_true"}:
        return 1
    # fallback heuristique
    if "fake" in s or "false" in s or "pants" in s or "barely" in s or "half" in s:
        return 0
    if "true" in s or "real" in s:
        return 1
    return np.nan

for df in [train, valid, test]:
    df["label_binary"] = df["label"].apply(to_binary_label)
    n_na = df["label_binary"].isna().sum()
    if n_na > 0:
        print(f"{n_na} lignes sans label binaire résolu dans {len(df)} lignes ; suppression")
        df.dropna(subset=["label_binary"], inplace=True)
    df["label_binary"] = df["label_binary"].astype(int)

print("État final :", train["label_binary"].isna().sum(), valid["label_binary"].isna().sum(), test["label_binary"].isna().sum())

# 0) Création du dataset HF
train_ds = Dataset.from_pandas(train[["statement_clean", "label_binary"]])
val_ds = Dataset.from_pandas(valid[["statement_clean", "label_binary"]])
test_ds = Dataset.from_pandas(test[["statement_clean", "label_binary"]])

# 1) tokenisation DistilBERT
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    tok = tokenizer(
        batch["statement_clean"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tok["labels"] = batch["label_binary"]
    return tok

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["statement_clean", "label_binary"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["statement_clean", "label_binary"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["statement_clean", "label_binary"])




train label uniques: {'half-true': 2114, 'false': 1995, 'mostly-true': 1962, 'true': 1676, 'barely-true': 1654, 'pants-fire': 839}
valid label uniques: {'false': 263, 'mostly-true': 251, 'half-true': 248, 'barely-true': 237, 'true': 169, 'pants-fire': 116}
test label uniques: {'half-true': 265, 'false': 249, 'mostly-true': 241, 'barely-true': 212, 'true': 208, 'pants-fire': 92}
État final : 0 0 0


Map: 100%|██████████| 1267/1267 [00:00<00:00, 9335.77 examples/s]


In [126]:
# 2) modèle DistilBERT
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="./bert_run",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
   
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"acc": accuracy_score(labels, preds), "f1": f1, "precision": p, "recall": r}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate(test_ds)
print("DistilBERT test metrics:", metrics)

preds = trainer.predict(test_ds)
yhat = preds.predictions.argmax(-1)
print("DistilBERT test accuracy:", accuracy_score(test["label_binary"].astype(int), yhat))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8097.89it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Jules\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloade

Epoch,Training Loss,Validation Loss,Acc,F1,Precision,Recall
1,0.627760,0.592774,0.671340,0.440318,0.497006,0.395238
2,0.592643,0.590899,0.684579,0.436718,0.525084,0.373810


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.55it/s]
C:\Users\Jules\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
C:\Users\Jules\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then d

RuntimeError: on_train_begin must be called before on_evaluate

In [127]:
df.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,...,pants_on_fire_counts,context,statement_len_words,statement_len_chars,nb_exclamation,nb_question,history_total,history_false_ratio,statement_clean,label_binary
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,...,18,Radio interview,11,68,0,0,143,0.333333,building a wall on the u.s.-mexico border will...,1
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,...,0,a news conference,12,63,0,0,3,0.250000,wisconsin is on pace to double the number of l...,0
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,...,61,comments on ABC's This Week.,10,51,0,0,326,0.535168,says john mccain has done nothing to help the ...,0
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,...,1,a radio show,13,85,0,0,7,0.250000,suzanne bonamici supports a plan that will cut...,0
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,...,7,a web video,23,127,0,0,23,0.583333,when asked by a reporter whether hes at the ce...,0


In [130]:
print(train_ds.column_names)
print(train_ds.features["labels"])
print(train_ds[0]["labels"])
print(train["label"].unique())
print(train["label"].isna().sum())

['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Value('int64')
0
['false' 'half-true' 'mostly-true' 'true' 'barely-true' 'pants-fire']
0


In [131]:
# 1. TF-IDF + modèle linéaire
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = vectorizer.fit_transform(train['statement_clean'])
X_valid_tfidf = vectorizer.transform(valid['statement_clean'])
X_test_tfidf  = vectorizer.transform(test['statement_clean'])

y_train = train['label_binary'].astype(int)
y_valid = valid['label_binary'].astype(int)
y_test  = test['label_binary'].astype(int)

clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
clf_tfidf.fit(X_train_tfidf, y_train)

for name, X, y in [('train', X_train_tfidf, y_train), ('valid', X_valid_tfidf, y_valid), ('test', X_test_tfidf, y_test)]:
    y_pred = clf_tfidf.predict(X)
    p, r, f1, _ = precision_recall_fscore_support(y, y_pred, average='binary', zero_division=0)
    print(f"TF-IDF {name}: acc={accuracy_score(y, y_pred):.4f}, f1={f1:.4f}, p={p:.4f}, r={r:.4f}")

TF-IDF train: acc=0.7612, f1=0.5370, p=0.8631, r=0.3898
TF-IDF valid: acc=0.6745, f1=0.3010, p=0.5056, r=0.2143
TF-IDF test: acc=0.6511, f1=0.2871, p=0.5205, r=0.1982


In [132]:
# 2. GloVe moyenne + modèle linéaire
import numpy as np
import gensim.downloader as api

# téléchargement requis une seule fois, possible dépendance réseau
glove = api.load('glove-wiki-gigaword-50')  # 50-d embeddings


def sentence_embedding(text, model=glove, dim=50):
    tokens = str(text).lower().split()
    vecs = [model[w] for w in tokens if w in model]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0)

X_train_glove = np.vstack([sentence_embedding(t) for t in train['statement_clean']])
X_valid_glove = np.vstack([sentence_embedding(t) for t in valid['statement_clean']])
X_test_glove  = np.vstack([sentence_embedding(t) for t in test['statement_clean']])

clf_glove = LogisticRegression(max_iter=1000, random_state=42)
clf_glove.fit(X_train_glove, y_train)

for name, X, y in [('train', X_train_glove, y_train), ('valid', X_valid_glove, y_valid), ('test', X_test_glove, y_test)]:
    y_pred = clf_glove.predict(X)
    p, r, f1, _ = precision_recall_fscore_support(y, y_pred, average='binary', zero_division=0)
    print(f"GloVe {name}: acc={accuracy_score(y, y_pred):.4f}, f1={f1:.4f}, p={p:.4f}, r={r:.4f}")

GloVe train: acc=0.6569, f1=0.2427, p=0.5624, r=0.1548
GloVe valid: acc=0.6877, f1=0.2642, p=0.5760, r=0.1714
GloVe test: acc=0.6559, f1=0.2270, p=0.5565, r=0.1425


In [133]:
# 3. DistilBERT (déjà entraîné ci-dessus)
print('DistilBERT est déjà entraîné, appel sur test :')
preds = trainer.predict(test_ds)
yhat = preds.predictions.argmax(-1)

print('DistilBERT test:', accuracy_score(y_test, yhat), 'f1', precision_recall_fscore_support(y_test, yhat, average='binary', zero_division=0)[2])

DistilBERT est déjà entraîné, appel sur test :


C:\Users\Jules\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DistilBERT test: 0.6787687450670876 f1 0.48284625158831
